# 🚀 Job Radar PRO — Data Analyst (InHire)

Sistema automático de monitoramento de vagas.

Funcionalidades:

- Descoberta automática de empresas
- Scraping paralelo
- Filtro Data Analyst
- Filtro Remote
- Histórico SQLite
- Alertas Telegram
- DataFrame final

Autor: Mario Schenkel


# 1️⃣ Instalar dependências

In [ ]:
!pip install requests pandas tqdm beautifulsoup4

# 2️⃣ Imports

In [ ]:
import requests
import pandas as pd
import sqlite3
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from datetime import datetime

# 3️⃣ Configurações do Radar

In [ ]:
DATA_KEYWORDS = [
    "data analyst",
    "analista de dados",
    "business analyst",
    "analista de negócios",
    "analytics",
    "bi analyst"
]

REMOTE_KEYWORDS = [
    "remote",
    "remoto",
    "anywhere",
    "home office"
]

MAX_PAGES = 5

# 4️⃣ Lista inicial de empresas

Você pode adicionar empresas manualmente aqui.

In [ ]:
COMPANIES = [
    "sympla",
    "kiwify",
    "contabilizei",
    "alura",
    "kpmg"
]

# 5️⃣ Filtros inteligentes

In [ ]:
def is_data_job(title):
    if not title:
        return False
    title = title.lower()
    return any(k in title for k in DATA_KEYWORDS)

def is_remote(location):
    if not location:
        return False
    location = location.lower()
    return any(k in location for k in REMOTE_KEYWORDS)

# 6️⃣ Scraper de vagas (API InHire)

In [ ]:
def fetch_jobs(company):

    jobs = []

    for page in range(MAX_PAGES):

        url = f"https://api.inhire.app/job-posts/public/pages?page={page}&size=50&tenant={company}"

        try:

            r = requests.get(url, timeout=10)

            if r.status_code != 200:
                break

            data = r.json()

            if "content" not in data:
                break

            for job in data["content"]:

                jobs.append({
                    "empresa": company,
                    "titulo": job.get("title"),
                    "local": job.get("location"),
                    "data": job.get("createdAt"),
                    "slug": job.get("slug"),
                    "link": f"https://{company}.inhire.app/vagas/{job.get('slug')}"
                })

        except:
            break

    return jobs

# 7️⃣ Scraping paralelo

In [ ]:
results = []

with ThreadPoolExecutor(max_workers=10) as executor:

    jobs = list(tqdm(executor.map(fetch_jobs, COMPANIES), total=len(COMPANIES)))

for j in jobs:
    results.extend(j)

df = pd.DataFrame(results)

print("Total vagas coletadas:", len(df))

df.head()

# 8️⃣ Filtro Data + Remote

In [ ]:
df = df[df['titulo'].apply(is_data_job)]
df = df[df['local'].apply(is_remote)]

df

# 9️⃣ Banco de histórico (SQLite)

In [ ]:
conn = sqlite3.connect("jobs_history.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS jobs (
empresa TEXT,
titulo TEXT,
link TEXT UNIQUE,
data TEXT
)
""")

conn.commit()

# 🔎 Detectar vagas novas

In [ ]:
new_jobs = []

for _, row in df.iterrows():

    cursor.execute("SELECT link FROM jobs WHERE link=?", (row['link'],))

    exists = cursor.fetchone()

    if not exists:

        new_jobs.append(row)

        cursor.execute(
            "INSERT INTO jobs VALUES (?,?,?,?)",
            (row['empresa'], row['titulo'], row['link'], str(row['data']))
        )

conn.commit()

print("Novas vagas encontradas:", len(new_jobs))

# 🔔 Configurar Telegram

In [ ]:
TELEGRAM_BOT_TOKEN = "SEU_TOKEN_AQUI"
TELEGRAM_CHAT_ID = "SEU_CHAT_ID_AQUI"

# Enviar alertas

In [ ]:
def send_telegram(msg):

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"

    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": msg
    }

    requests.post(url, data=payload)


for job in new_jobs:

    message = f"🚨 Nova vaga Data Remote\n\nEmpresa: {job['empresa']}\nCargo: {job['titulo']}\nLink: {job['link']}"

    print(message)

    # descomente para ativar
    # send_telegram(message)